In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import mph
import mphsweepkit as msk

In [3]:
# Start the COMSOL client
client = mph.start()

In [4]:
# Load the model
model = client.load('toroid_cross_section_solved.mph')

Load an already solved CascadedSweepModel

In [5]:
csm = msk.CascadedSweepModel(model, 'Study on Cross-Sections', show_param_names=True)

Initialized CascadedSweepModel
Study name: Study on Cross-Sections
Sweep Structure:
    - Geometry Sweep (BatchSweep) -> 'h_c', 'd_o', 'd_i'
      - Material Sweep (MaterialSweep) -> 'matsw.comp1.core1'
        - Excitation Sweep (Parametric) -> 'b_mean'
          - Frequency Sweep (Frequency)
Loop names: ['Geometry Sweep', 'Material Sweep', 'Excitation Sweep', 'Frequency Sweep']
Loop lengths: [3, 1, 4, 11]
Added 'geometry_idx' and 'internal_idx' columns to input_data. New shape: (132, 8)
--------------------------------
Data updated from MPh-model.
Input data shape: (132, 8)
Reset output data to shape of the input data: (132, 0)
Combined shape: (132, 8)


Post-processing of global equations

In [6]:
# Load the customized post-processing expressions from a JSON file
post_processing_exprs = msk.load_post_processing_exprs("global_post_processing_expressions.json", print_info=False)

# Perform post-processing of global data
csm.post_process_globals(post_processing_exprs)

# Show the first few rows of the combined data
csm.combined_data.head()

name,h_c,d_o,d_i,matsw.comp1.core1,b_mean,freq,geometry_idx,internal_idx,b_eff,h_eff,mu_r,p_loss,p_mag,p_el
unit,mm,mm,mm,,mT,kHz,,,T,A/m,1,W /m^3,W /m^3,W /m^3
group,Geometry Sweep,Geometry Sweep,Geometry Sweep,Material Sweep,Excitation Sweep,Frequency Sweep,Indexing,Indexing,post-processing,post-processing,post-processing,post-processing,post-processing,post-processing
0,10.0,14.0,10.0,2.0,25.0,100.0,0,0,0.025,11.195014+ 0.130984j,1776.830712- 20.789352j,1037.937937,1026.598014,11.339924
1,10.0,14.0,10.0,2.0,25.0,200.0,0,1,0.025,11.135700+ 0.166083j,1786.142145- 26.639278j,2631.470363,2567.828126,63.642237
2,10.0,14.0,10.0,2.0,25.0,300.0,0,2,0.025,11.073786+ 0.204355j,1795.916553- 33.141699j,4854.808797,4672.006176,182.802621
3,10.0,14.0,10.0,2.0,25.0,400.0,0,3,0.025,11.009312+ 0.253985j,1806.087965- 41.666424j,8040.966455,7647.764785,393.201670
4,10.0,14.0,10.0,2.0,25.0,500.0,0,4,0.025,10.942201+ 0.320011j,1816.578483- 53.126937j,12657.199608,11939.027765,718.171844


In [7]:
# Save the input and output dataframes to CSV files
csm.save_global_data()

Saved result data to: X:\Till_data\repositories\MPhSweepKit\examples\studies\toroid_cross_section\global_data


Post-processing of fields on a selected domain

In [8]:
# All available selections can be printed with
# csm.print_available_selections()

# Select a domain selection for post-processing.
# The selection name must exist in the COMSOL model:
selection_name = "Toroid"
selection_type = "dom"
selection_domain_tag = csm._get_selection_tag(selection_name, selection_type="dom")
print(f"Selected domain tag: {selection_domain_tag}")

# Choose a name for the dataset that will be created from the selection:
selection_dataset_name = "Solution Core"

# Create a dataset from the selection
csm.create_dataset_selection(selection_name, selection_type, selection_dataset_name)

# A dataset can be removed with
# csm.node_datasets.children()[-1].remove()

Selected domain tag: geom2_csel1_dom
Creating dataset 'Solution Core' from selection 'Toroid' of type 'dom'.


In [9]:
# Load customized post-processing expressions
post_processing_exprs = msk.load_post_processing_exprs("fields_post_processing_expressions.json", print_info=False)

# Compute and export the fields on all geometries in the selection dataset
# In the chosen sub-folder a new file will be created for each geometry in the selection dataset, containing the computed fields.
csm.export_fields_on_all_geometries(
    dataset_name=selection_dataset_name,
    post_processing_exprs=post_processing_exprs,
    sub_folder="Fields in Cross-Section"
)

Loop levels from COMSOL: [[1, 2, 3], [1], [1, 2, 3, 4], [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]]

Dataset to be exported:           'Solution Core'
    Check status of export nodes:
  Create export node:             'Magnetic Flux Density on Solution Core'
  Create export node:             'Electric Field on Solution Core'
  Create export node:             'Complex Permeability on Solution Core'
    Exported field data to:       'Fields in Cross-Section/geometry_0_Magnetic Flux Density.txt'
    Exported field data to:       'Fields in Cross-Section/geometry_0_Electric Field.txt'
    Exported field data to:       'Fields in Cross-Section/geometry_0_Complex Permeability.txt'

Dataset to be exported:           'Solution Core'
    Check status of export nodes:
        Overwrite export node:    'Magnetic Flux Density on Solution Core'
        Overwrite export node:    'Electric Field on Solution Core'
        Overwrite export node:    'Complex Permeability on Solution Core'
    Exported field d

Release Model

In [10]:
client.remove(model)
client.clear()
client.disconnect()